# CO_POF



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/Conformal_Oracle/blob/main/CO_POF/CO_POF.ipynb)



**Raw Percentage of Failures (POF) for 1% VaR across LLM architectures (GPT-3.5, GPT-4, GPT-4o)**



Published in: *Calibrating the Oracle — Distribution-Free Conformal Risk Measures for LLM-Based Financial Forecasting*



Section: Raw Calibration Diagnostics


In [ ]:
# Install dependencies (if running on Google Colab)

import sys

if 'google.colab' in sys.modules:

    !pip install -q numpy pandas matplotlib scipy


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

# ── Configuration ──
ALPHA_VAR = 0.01
F_CAL = 0.70
MODELS = ["GPT-3.5", "GPT-4", "GPT-4o"]
ASSETS = ["CRIX", "SP500", "SPGTCLTR", "stoxx", "cact", "gdaxi", "cbu", "ftse", "djci"]

# ── Helper: Kupiec POF test ──
def kupiec_test(realized, var_est, alpha=0.01):
    """Proportion of failures test (Kupiec 1995)."""
    mask = ~np.isnan(realized) & ~np.isnan(var_est)
    r, v = realized[mask], var_est[mask]
    N = len(r)
    if N == 0:
        return {'N': 0, 'x': 0, 'pi_hat': np.nan, 'p_value': np.nan}
    violations = r < -v
    x = int(violations.sum())
    pi_hat = x / N
    if x == 0 or x == N:
        p_val = 0.0 if abs(pi_hat - alpha) > 0.02 else 1.0
    else:
        lr = -2 * (x * np.log(alpha / pi_hat) + (N - x) * np.log((1 - alpha) / (1 - pi_hat)))
        p_val = 1 - stats.chi2.cdf(abs(lr), df=1)
    return {'N': N, 'x': x, 'pi_hat': pi_hat, 'p_value': p_val}

print("POF test computes the empirical violation rate π̂ = x/N")
print(f"Target: α = {ALPHA_VAR} (1%)")
print(f"Basel Green zone: ≤ 4 annual exceedances (π̂ ≤ 0.016)")
print(f"\nLoad your simulation data and realized returns, then call kupiec_test().")